# Reporting Views — Media & Telecom Analytics
Creates SQL views over Gold tables for Power BI Direct Lake connectivity: executive summary, churn risk, content performance, and ad revenue breakdown.

In [ ]:
# vw_executive_summary — top-line KPIs across plans and regions
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    plan_type,
    region,
    total_subscribers,
    active_subscribers,
    churned_count,
    churn_rate,
    total_mrr,
    arpu,
    avg_tenure_days,
    health_band,
    RANK() OVER (PARTITION BY plan_type ORDER BY churn_rate ASC) AS churn_rank_in_plan
FROM gold_subscriber_scorecards
""")
print('Created: vw_executive_summary')

In [ ]:
# vw_churn_risk — cohort churn analysis with at-risk revenue
spark.sql("""
CREATE OR REPLACE VIEW vw_churn_risk AS
SELECT
    tenure_bucket,
    plan_type,
    region,
    age_group,
    total_subs,
    churned,
    churn_rate,
    avg_tenure_days,
    total_mrr,
    at_risk_revenue
FROM gold_churn_analysis
ORDER BY churn_rate DESC
""")
print('Created: vw_churn_risk')

In [ ]:
# vw_content_performance — top content by views and completion for editorial teams
spark.sql("""
CREATE OR REPLACE VIEW vw_content_performance AS
SELECT
    cp.content_id,
    cc.title,
    cp.genre,
    cp.content_type,
    cp.language,
    cp.release_year,
    cp.total_views,
    cp.unique_viewers,
    cp.avg_watch_mins,
    cp.avg_completion_pct,
    cp.completion_rate,
    cp.avg_rating,
    RANK() OVER (PARTITION BY cp.genre ORDER BY cp.total_views DESC) AS rank_in_genre
FROM gold_content_performance cp
LEFT JOIN silver_content_catalog cc ON cp.content_id = cc.content_id
""")
print('Created: vw_content_performance')

In [ ]:
# vw_ad_revenue_breakdown — ad performance for monetisation reporting
spark.sql("""
CREATE OR REPLACE VIEW vw_ad_revenue_breakdown AS
SELECT
    ad_type,
    genre,
    ctr_band,
    total_impressions,
    total_clicks,
    ROUND(total_revenue, 2)  AS total_revenue,
    avg_cpm,
    avg_ctr,
    ecpm,
    RANK() OVER (PARTITION BY ad_type ORDER BY total_revenue DESC) AS revenue_rank
FROM gold_ad_revenue
""")
print('Created: vw_ad_revenue_breakdown')

In [ ]:
# Validate
views = ['vw_executive_summary', 'vw_churn_risk', 'vw_content_performance', 'vw_ad_revenue_breakdown']
for v in views:
    n = spark.sql(f'SELECT COUNT(*) AS n FROM {v}').collect()[0]['n']
    print(f'  {v:<35} {n:>6,} rows')